<a href="https://colab.research.google.com/github/aprameya-rao/BIS_Lab/blob/main/BIS_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Genetic Mutation Algorithm for Traffic managment

In [ ]:
import random

SIMULATION_TIME = 300
YELLOW_TIME = 3

MIN_GREEN = 20
MAX_GREEN = 60

POOL_SIZE = 30
GENERATIONS = 5

ADJUSTMENT_RATE = 0.1
TOURNAMENT_SIZE = 3

def generate_traffic(rate):
    arrivals = []
    for t in range(SIMULATION_TIME):
        if random.random() < rate:
            arrivals.append(t)
    return arrivals

def generate_all_traffic():
    traffic = {
        "North": generate_traffic(0.30),
        "South": generate_traffic(0.25),
        "East": generate_traffic(0.15),
        "West": generate_traffic(0.20)
    }
    return traffic

def get_signal_state(time, ns_green, ew_green):
    cycle = ns_green + YELLOW_TIME + ew_green + YELLOW_TIME
    t = time % cycle
    if t < ns_green:
        return "NS_GREEN"
    elif t < ns_green + YELLOW_TIME:
        return "NS_YELLOW"
    elif t < ns_green + YELLOW_TIME + ew_green:
        return "EW_GREEN"
    else:
        return "EW_YELLOW"

def simulate(timing_plan, traffic):
    ns_green = timing_plan[0]
    ew_green = timing_plan[1]

    # Queues
    queues = {
        "North": [],
        "South": [],
        "East": [],
        "West": []
    }

    total_waiting_time = 0
    total_queue_time = 0
    stopped_vehicles = 0

    # Store arrival index for each road
    indexes = {
        "North": 0,
        "South": 0,
        "East": 0,
        "West": 0
    }

    for time in range(SIMULATION_TIME):
        for road in traffic:
            arrivals = traffic[road]
            while (
                indexes[road] < len(arrivals)
                and arrivals[indexes[road]] == time
            ):
                queues[road].append(time)
                indexes[road] += 1

        signal = get_signal_state(
            time,
            ns_green,
            ew_green
        )

        if signal == "NS_GREEN":
            active_roads = ["North", "South"]
        elif signal == "EW_GREEN":
            active_roads = ["East", "West"]
        else:
            active_roads = []

        for road in active_roads:
            if len(queues[road]) > 0:
                arrival_time = queues[road].pop(0)
                waiting_time = time - arrival_time
                total_waiting_time += waiting_time

        for road in queues:
            queue_length = len(queues[road])
            total_queue_time += queue_length
            if queue_length > 0:
                stopped_vehicles += queue_length

    cost = (
        total_waiting_time
        + 0.5 * total_queue_time
        + 0.1 * stopped_vehicles
    )

    efficiency_score = 1 / (1 + cost)

    return efficiency_score, {
        "waiting_time": total_waiting_time,
        "queue_time": total_queue_time,
        "stopped_vehicles": stopped_vehicles
    }

def create_random_plan():
    ns_green = random.randint(
        MIN_GREEN,
        MAX_GREEN
    )
    ew_green = random.randint(
        MIN_GREEN,
        MAX_GREEN
    )
    return [ns_green, ew_green]

def create_initial_plans():
    return [
        create_random_plan()
        for _ in range(POOL_SIZE)
    ]

def select_plan(plan_pool, traffic):
    tournament = random.sample(
        plan_pool,
        TOURNAMENT_SIZE
    )
    tournament.sort(
        key=lambda x: simulate(x, traffic)[0],
        reverse=True
    )
    return tournament[0]

def combine_plans(plan1, plan2):
    combined = [
        plan1[0],
        plan2[1]
    ]
    return combined

def adjust_plan(timing_plan):
    if random.random() < ADJUSTMENT_RATE:
        signal_dimension = random.randint(0, 1)
        timing_plan[signal_dimension] += random.randint(-5, 5)
        timing_plan[signal_dimension] = max(
            MIN_GREEN,
            min(MAX_GREEN, timing_plan[signal_dimension])
        )
    return timing_plan

def traffic_plan_optimizer(traffic):
    plan_pool = create_initial_plans()

    best_plan = None
    best_efficiency = -float("inf")

    history = []

    for generation in range(GENERATIONS):
        evaluated = []
        for timing_plan in plan_pool:
            efficiency, stats = simulate(
                timing_plan,
                traffic
            )
            evaluated.append(
                (efficiency, timing_plan, stats)
            )

        evaluated.sort(
            key=lambda x: x[0],
            reverse=True
        )

        current_best = evaluated[0]

        if current_best[0] > best_efficiency:
            best_efficiency = current_best[0]
            best_plan = current_best[1].copy()

        history.append(best_efficiency)

        print(
            f"Generation {generation + 1:02d} | "
            f"Best Timing Plan = {current_best[1]} | "
            f"Efficiency Score = {current_best[0]:.8f}"
        )

        new_plan_pool = []

        # Elite preservation
        new_plan_pool.append(
            evaluated[0][1].copy()
        )

        while len(new_plan_pool) < POOL_SIZE:
            parent_plan1 = select_plan(
                plan_pool,
                traffic
            )
            parent_plan2 = select_plan(
                plan_pool,
                traffic
            )
            child_plan = combine_plans(
                parent_plan1,
                parent_plan2
            )
            child_plan = adjust_plan(child_plan)
            new_plan_pool.append(child_plan)

        plan_pool = new_plan_pool

    return best_plan, history

if __name__ == "__main__":
    random.seed(42)

    # Generate traffic
    traffic = generate_all_traffic()

    print("\nTraffic generated:")
    for road in traffic:
        print(
            road,
            "vehicles =",
            len(traffic[road])
        )

    print("\nRunning Traffic Plan Optimizer...\n")

    best_solution, history = traffic_plan_optimizer(
        traffic
    )
    fitness, stats = simulate(
        best_solution,
        traffic
    )

    print("\n==============================")
    print("OPTIMAL SIGNAL TIMING")
    print("==============================")
    print("North-South Green:", best_solution[0], "seconds")
    print("East-West Green:", best_solution[1], "seconds")
    print("Yellow:", YELLOW_TIME, "seconds")

    print("\nPerformance:")
    print("Total waiting time:", stats["waiting_time"])
    print("Total queue time:", stats["queue_time"])
    print("Stopped vehicles:", stats["stopped_vehicles"])


Traffic generated:
North vehicles = 105
South vehicles = 72
East vehicles = 35
West vehicles = 65

Running Traffic Plan Optimizer...

Generation 01 | Best Timing Plan = [30, 24] | Efficiency Score = 0.00021550
Generation 02 | Best Timing Plan = [25, 24] | Efficiency Score = 0.00021900
Generation 03 | Best Timing Plan = [30, 20] | Efficiency Score = 0.00023514
Generation 04 | Best Timing Plan = [30, 20] | Efficiency Score = 0.00023514
Generation 05 | Best Timing Plan = [25, 20] | Efficiency Score = 0.00025084

OPTIMAL SIGNAL TIMING
North-South Green: 25 seconds
East-West Green: 20 seconds
Yellow: 3 seconds

Performance:
Total waiting time: 2437
Total queue time: 2581
Stopped vehicles: 2581


Particle Swarm Optimisation

In [ ]:
import numpy as np
import random

SWARM_SIZE = 20
ITERATIONS = 3
W = 0.5
C1 = 1.5
C2 = 2.0

class Particle:
    def __init__(self):
        self.position = np.array([
            float(random.randint(MIN_GREEN, MAX_GREEN)),
            float(random.randint(MIN_GREEN, MAX_GREEN))
        ])
        self.velocity = np.array([
            random.uniform(-5, 5),
            random.uniform(-5, 5)
        ])
        self.best_position = np.copy(self.position)
        self.best_fitness = -float('inf')

def pso_optimize(traffic):
    swarm = [Particle() for _ in range(SWARM_SIZE)]

    global_best_position = None
    global_best_fitness = -float('inf')

    history = []

    for it in range(ITERATIONS):
        for particle in swarm:
            chromosome_representation = [int(round(particle.position[0])), int(round(particle.position[1]))]
            fitness, _ = simulate(chromosome_representation, traffic)
            if fitness > particle.best_fitness:
                particle.best_fitness = fitness
                particle.best_position = np.copy(particle.position)
            if fitness > global_best_fitness:
                global_best_fitness = fitness
                global_best_position = np.copy(particle.position)
        for particle in swarm:
            r1, r2 = random.random(), random.random()
            cognitive = C1 * r1 * (particle.best_position - particle.position)
            social = C2 * r2 * (global_best_position - particle.position)

            particle.velocity = W * particle.velocity + cognitive + social
            particle.position = particle.position + particle.velocity
            particle.position[0] = max(MIN_GREEN, min(MAX_GREEN, particle.position[0]))
            particle.position[1] = max(MIN_GREEN, min(MAX_GREEN, particle.position[1]))

        history.append(global_best_fitness)
        best_rounded = [int(round(global_best_position[0])), int(round(global_best_position[1]))]
        print(f"Iteration {it+1:02d} | Global Best Position = {best_rounded} | Fitness = {global_best_fitness:.8f}")

    return [int(round(global_best_position[0])), int(round(global_best_position[1]))], history

# Run the PSO algorithm on the generated traffic
print("\nRunning Particle Swarm Optimization...\n")
best_pso_solution, pso_history = pso_optimize(traffic)

pso_fitness, pso_stats = simulate(best_pso_solution, traffic)

print("\n==============================")
print("OPTIMAL SIGNAL TIMING (PSO)")
print("==============================")
print("North-South Green:", best_pso_solution[0], "seconds")
print("East-West Green:", best_pso_solution[1], "seconds")
print("Yellow:", YELLOW_TIME, "seconds")
print("\nPerformance:")
print("Total waiting time:", pso_stats["waiting_time"])
print("Total queue time:", pso_stats["queue_time"])
print("Stopped vehicles:", pso_stats["stopped_vehicles"])


Running Particle Swarm Optimization...

Iteration 01 | Global Best Position = [39, 21] | Fitness = 0.00022239
Iteration 02 | Global Best Position = [30, 20] | Fitness = 0.00023514
Iteration 03 | Global Best Position = [22, 20] | Fitness = 0.00025818

OPTIMAL SIGNAL TIMING (PSO)
North-South Green: 22 seconds
East-West Green: 20 seconds
Yellow: 3 seconds

Performance:
Total waiting time: 2404
Total queue time: 2447
Stopped vehicles: 2447


Ant Colony Optimisation for Network Packet Routing

In [ ]:
import numpy as np
import random

NUM_NODES = 6
START_NODE = 0
DEST_NODE = 5

NUM_ANTS = 25
NUM_ITERATIONS = 5

ALPHA = 1.0
BETA = 2.0
EVAPORATION = 0.3
Q = 10.0

INF = 10000000
distance_matrix = np.array([
    [0,   10,   15,   INF, INF, INF],
    [INF,   0,   INF,   22,  23, INF],
    [0,   0,   0,   18,   10,  INF],
    [INF, INF,   INF,   0,   INF,   12  ],
    [INF, INF, INF,  INF,   0,   15  ],
    [INF, INF, INF, INF,   INF,   0  ]
])

pheromones = np.ones((NUM_NODES, NUM_NODES)) * 0.1

def get_neighbors(node):
    neighbors = []
    for idx, dist in enumerate(distance_matrix[node]):
        if dist > 0 and dist != INF:
            neighbors.append(idx)
    return neighbors

class Ant:
    def __init__(self, start):
        self.start = start
        self.current_node = start
        self.path = [start]
        self.path_distance = 0.0
        self.visited = {start}

    def select_next_node(self):
        neighbors = get_neighbors(self.current_node)
        unvisited_neighbors = [n for n in neighbors if n not in self.visited]

        if not unvisited_neighbors:
            return False

        probabilities = []
        for neighbor in unvisited_neighbors:
            pheromone = pheromones[self.current_node][neighbor]
            heuristic = 1.0 / distance_matrix[self.current_node][neighbor]
            prob = (pheromone ** ALPHA) * (heuristic ** BETA)
            probabilities.append(prob)

        sum_prob = sum(probabilities)
        if sum_prob == 0:
            probabilities = [1.0 / len(unvisited_neighbors)] * len(unvisited_neighbors)
        else:
            probabilities = [p / sum_prob for p in probabilities]

        next_node = np.random.choice(unvisited_neighbors, p=probabilities)

        self.path_distance += distance_matrix[self.current_node][next_node]
        self.current_node = next_node
        self.path.append(next_node)
        self.visited.add(next_node)
        return True

best_path = None
best_distance = INF

print("Running Ant Colony Optimization for Packet Routing...\n")

for iteration in range(NUM_ITERATIONS):
    ants = []
    for _ in range(NUM_ANTS):
        ant = Ant(START_NODE)
        reached_dest = False
        while ant.current_node != DEST_NODE:
            success = ant.select_next_node()
            if not success:
                break
        if ant.current_node == DEST_NODE:
            ants.append(ant)
            if ant.path_distance < best_distance:
                best_distance = ant.path_distance
                best_path = list(ant.path)

    pheromones *= (1 - EVAPORATION)

    for ant in ants:
        pheromone_deposit = Q / ant.path_distance
        for i in range(len(ant.path) - 1):
            u, v = ant.path[i], ant.path[i+1]
            pheromones[u][v] += pheromone_deposit
            pheromones[v][u] += pheromone_deposit

    print(f"IT {iteration+1:02d} | Ants reached : {len(ants)}/{NUM_ANTS} | Shortest routing path : {best_distance if best_distance != INF else 'N/A'}")

print("\n==============================")
print("OPTIMAL ROUTE (ACO)")
print("==============================")
print("Shortest Path Found:", " -> ".join(map(str, best_path)) if best_path else "None")
print("Path Distance:", best_distance)


Running Ant Colony Optimization for Packet Routing...

IT 01 | Ants reached : 25/25 | Shortest routing path : 40.0
IT 02 | Ants reached : 25/25 | Shortest routing path : 40.0
IT 03 | Ants reached : 25/25 | Shortest routing path : 40.0
IT 04 | Ants reached : 25/25 | Shortest routing path : 40.0
IT 05 | Ants reached : 25/25 | Shortest routing path : 40.0

OPTIMAL ROUTE (ACO)
Shortest Path Found: 0 -> 2 -> 4 -> 5
Path Distance: 40.0


Cuckoo Search - Network Optimisation

In [1]:
import numpy as np
import random
import math

INF = 100000.0
network_matrix = np.array([
    [0,   10,  15,  INF, INF, INF],
    [INF, 0,   INF, 22,  23,  INF],
    [0,   0,   0,   18,  10,  INF],
    [INF, INF, INF, 0,   INF, 12 ],
    [INF, INF, INF, INF, 0,   15 ],
    [INF, INF, INF, INF, INF, 0  ]
])

NUM_NODES = 6
START_NODE = 0
DEST_NODE = 5

POP_SIZE = 15
PA = 0.25
GENERATIONS = 30
LEVY_FLIGHT_BETA = 1.5

def get_valid_neighbors(node):
    neighbors = []
    for idx, weight in enumerate(network_matrix[node]):
        if weight > 0 and weight != INF:
            neighbors.append(idx)
    return neighbors

def generate_random_path(start, end):
    path = [start]
    current = start
    visited = {start}

    while current != end:
        neighbors = [n for n in get_valid_neighbors(current) if n not in visited]
        if not neighbors:
            return generate_random_path(start, end)
        next_node = random.choice(neighbors)
        path.append(next_node)
        visited.add(next_node)
        current = next_node
    return path

def calculate_path_cost(path):
    cost = 0.0
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        weight = network_matrix[u][v]
        if weight == INF:
            return INF
        cost += weight
    return cost

def levy_flight(step_size=1.0):
    sigma_u = (math.gamma(1 + LEVY_FLIGHT_BETA) * math.sin(math.pi * LEVY_FLIGHT_BETA / 2) /
               (math.gamma((1 + LEVY_FLIGHT_BETA) / 2) * LEVY_FLIGHT_BETA * 2**((LEVY_FLIGHT_BETA - 1) / 2)))**(1 / LEVY_FLIGHT_BETA)
    u = np.random.normal(0, sigma_u)
    v = np.random.normal(0, 1)
    step = u / (abs(v)**(1 / LEVY_FLIGHT_BETA))
    return step_size * step

def mutate_path(path, step_length):
    if len(path) <= 2:
        return generate_random_path(START_NODE, DEST_NODE)
    mutation_idx = random.randint(1, len(path) - 2)
    partial_start = path[:mutation_idx]
    current = partial_start[-1]

    try:
        suffix = generate_random_path(current, DEST_NODE)
        new_path = partial_start[:-1] + suffix
        return new_path
    except:
        return generate_random_path(START_NODE, DEST_NODE)

nests = [generate_random_path(START_NODE, DEST_NODE) for _ in range(POP_SIZE)]
fitness = [calculate_path_cost(nest) for nest in nests]

best_idx = np.argmin(fitness)
best_nest = nests[best_idx]
best_cost = fitness[best_idx]

print("Running Cuckoo Search for Shortest Path Routing...\n")

for gen in range(GENERATIONS):
    step = levy_flight()
    random_nest_idx = random.randint(0, POP_SIZE - 1)
    cuckoo_path = mutate_path(nests[random_nest_idx], step)
    cuckoo_cost = calculate_path_cost(cuckoo_path)
    target_idx = random.randint(0, POP_SIZE - 1)
    if cuckoo_cost < fitness[target_idx]:
        nests[target_idx] = cuckoo_path
        fitness[target_idx] = cuckoo_cost
    for i in range(POP_SIZE):
        if random.random() < PA:
            nests[i] = generate_random_path(START_NODE, DEST_NODE)
            fitness[i] = calculate_path_cost(nests[i])
    current_best_idx = np.argmin(fitness)
    if fitness[current_best_idx] < best_cost:
        best_cost = fitness[current_best_idx]
        best_nest = list(nests[current_best_idx])

    if (gen + 1) % 5 == 0 or gen == 0:
        print(f"Gen {gen+1:02d} | Best Route Cost: {best_cost} | Route: {' -> '.join(map(str, best_nest))}")

print("\n==============================")
print("OPTIMAL ROUTE (CUCKCO SEARCH)")
print("==============================")
print("Shortest Path Found:", " -> ".join(map(str, best_nest)) if best_nest else "None")
print("Path Distance:", best_cost)

Running Cuckoo Search for Shortest Path Routing...

Gen 01 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5
Gen 05 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5
Gen 10 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5
Gen 15 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5
Gen 20 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5
Gen 25 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5
Gen 30 | Best Route Cost: 40.0 | Route: 0 -> 2 -> 4 -> 5

OPTIMAL ROUTE (CUCKCO SEARCH)
Shortest Path Found: 0 -> 2 -> 4 -> 5
Path Distance: 40.0


Grey Wolf Optimisation - Municipal waste Collection Optimiser

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math

np.random.seed(42)
random.seed(42)

NUM_BINS = 12
DEPOT = [50, 50]
bin_coords = np.random.rand(NUM_BINS, 2) * 100
locations = np.vstack([DEPOT, bin_coords])
total_locations = len(locations)
def get_distance_matrix(locs):
    n = len(locs)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            matrix[i, j] = np.linalg.norm(locs[i] - locs[j])
    return matrix

dist_matrix = get_distance_matrix(locations)
def calculate_tour_distance(tour):
    distance = 0
    current = 0  # Depot index
    for next_node in tour:
        distance += dist_matrix[current, next_node]
        current = next_node
    distance += dist_matrix[current, 0]  # Back to depot
    return distance

NUM_WOLVES = 20
MAX_ITER = 100
DIM = NUM_BINS
wolves_pos = np.random.rand(NUM_WOLVES, DIM)

def get_tour_from_position(position):
    sorted_indices = np.argsort(position)
    tour = sorted_indices + 1
    return tour
alpha_pos = np.zeros(DIM)
alpha_score = float('inf')

beta_pos = np.zeros(DIM)
beta_score = float('inf')

delta_pos = np.zeros(DIM)
delta_score = float('inf')
history = []

print("Initializing Grey Wolf Optimizer for Waste Collection...\n")

for iteration in range(MAX_ITER):
    for i in range(NUM_WOLVES):
        tour = get_tour_from_position(wolves_pos[i])
        fitness = calculate_tour_distance(tour)

        if fitness < alpha_score:
            delta_score = beta_score
            delta_pos = beta_pos.copy()
            beta_score = alpha_score
            beta_pos = alpha_pos.copy()
            alpha_score = fitness
            alpha_pos = wolves_pos[i].copy()

        elif fitness < beta_score:
            delta_score = beta_score
            delta_pos = beta_pos.copy()
            beta_score = fitness
            beta_pos = wolves_pos[i].copy()

        elif fitness < delta_score:
            delta_score = fitness
            delta_pos = wolves_pos[i].copy()
    a = 2 - iteration * (2 / MAX_ITER)
    for i in range(NUM_WOLVES):
        for j in range(DIM):
            r1, r2 = random.random(), random.random()
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = abs(C1 * alpha_pos[j] - wolves_pos[i, j])
            X1 = alpha_pos[j] - A1 * D_alpha
            r1, r2 = random.random(), random.random()
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = abs(C2 * beta_pos[j] - wolves_pos[i, j])
            X2 = beta_pos[j] - A2 * D_beta
            r1, r2 = random.random(), random.random()
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = abs(C3 * delta_pos[j] - wolves_pos[i, j])
            X3 = delta_pos[j] - A3 * D_delta
            wolves_pos[i, j] = (X1 + X2 + X3) / 3
        wolves_pos[i] = np.clip(wolves_pos[i], 0, 1)

    history.append(alpha_score)
    if (iteration + 1) % 10 == 0 or iteration == 0:
        print(f"Iteration {iteration+1:03d} | Best Tour Distance: {alpha_score:.2f} km")
best_tour = get_tour_from_position(alpha_pos)
full_route = [0] + list(best_tour) + [0]

print("\n==============================================")
print("OPTIMAL WASTE COLLECTION ROUTE (GWO)")
print("==============================================")
print("Best Collection Sequence (Depot -> Bins -> Depot):")
print(" -> ".join(map(str, full_route)))
print(f"Total Optimal Path Length: {alpha_score:.2f} km")


Initializing Grey Wolf Optimizer for Waste Collection...

Iteration 001 | Best Tour Distance: 516.47 km
Iteration 010 | Best Tour Distance: 418.74 km
Iteration 020 | Best Tour Distance: 418.74 km
Iteration 030 | Best Tour Distance: 418.74 km
Iteration 040 | Best Tour Distance: 418.74 km
Iteration 050 | Best Tour Distance: 418.31 km
Iteration 060 | Best Tour Distance: 418.31 km
Iteration 070 | Best Tour Distance: 418.31 km
Iteration 080 | Best Tour Distance: 418.31 km
Iteration 090 | Best Tour Distance: 418.31 km
Iteration 100 | Best Tour Distance: 418.31 km

OPTIMAL WASTE COLLECTION ROUTE (GWO)
Best Collection Sequence (Depot -> Bins -> Depot):
0 -> 2 -> 7 -> 12 -> 9 -> 4 -> 6 -> 1 -> 5 -> 10 -> 8 -> 3 -> 11 -> 0
Total Optimal Path Length: 418.31 km
